# EDA/SDA 1 (Lean): Field Comparison and Scope

PredictLeads × TripleTen externship · Jacob Colombo

## Purpose

Clean PredictLeads' full job-openings file (every listing first seen March 30 – September 30, 2026), compare fields, cut to the scope (**the roles TripleTen trains for, plus adjacent tech roles**) and explore it.

- **Q1 (section 3):** how do computer and mathematical roles compare with every other field?
- **Q2 (section 5):** which factors should drive prioritization for a career changer?
- **Q3 (section 6):** what time information can prioritization rely on?

Fix or flag, never drop, except data errors; aggregates only, never rows. **"Still open"** means last seen within 2 days of September 30.

**Hands off (section 7):** the in-scope table (every field) and its descriptions, the occupation summary, and company activity.

| Notebook | Question | Hands off |
|:--|:--|:--|
| **EDA_SDA_1** | **What's in the data, how does tech compare to other fields, and what's in scope?** | **Scope table, descriptions, occupation summary, company activity** |
| EDA_SDA_2 | Which scope listings do we keep, and how do they behave? | Scope table, non-text features |
| EDA_SDA_3 | What do the descriptions say? Skills mapped to O*NET | Scope table + text features |
| EDA_SDA_4 | Which features hold up? Statistics, confounders, the adjusted Cox check | Modeling-ready table |
| TimeSeries | When do listings appear and close? | Timing features |
| Profiles | Who really changed into or within tech or math, and how? Career-switch rules on the public résumé data (frozen) | The 2,813 career changers, and the table ML_Fit trains on |
| ML_Fit | Which roles and listings fit a career changer? Keyword → TF-IDF → BGE → RRF, skills-based matching, a trained layer on real career moves | Fit scores, role discovery, the compass |
| ML_Rank | How should listings be ranked? The listing timing model, combined with fit into the priority queue | The priority queue algorithm |
| Evaluate | How well do the app's rankers do on real career moves? `rrf`, `mlrank` and the combined model | The ranker comparison |

Same table as [`../PROJECT.md`](../PROJECT.md) ("The notebooks"), kept in step with it.

### About this Lean draft

A plain-pandas rewrite of this notebook: every cell uses only pandas/numpy/matplotlib and the project's real
data-loading code (`src/pl_data.py`, `src/rows_table.py`, `src/stream_full.py`) - no shared formatting or
summary-table helpers (`src/eda_helpers.py`). The goal is code that can be explained line by line, the way a
TripleTen project can, even where that costs more lines than the original. This will replace `EDA_SDA_1`
once it reruns clean with the same results. It is on the pitch's critical path, not a side project: Jacob
presents these findings, so he has to be able to explain every cell that produces them. The Lean rewrites of
the other notebooks in the chain wait until after the pitch (`../ANALYSIS.md`). The standing rules
behind every call made here (what gets inlined, what stays a shared-file call, how a dropped helper gets
checked before it's trusted) are written down once in `docs/lean_conventions.md`, not re-argued per notebook.

Covers all eight sections (1-8); see the closing note at the end for where it differs from `EDA_SDA_1` and what's left
before it replaces it.

## 1. Load and build

1b–1d show the raw structure on a 200,000-document sample, then flatten it (`data/flat/`). 1e builds the full file (13.4M listings) from `data/raw/` (`python -m fetch.get_predictleads`) or S3 (`src/stream_full.py`). `BUILD_MODE`: **bulk** (about 35 minutes, 16 GB of RAM), **stream** (a few hours, for an 8 GB machine) or **auto**; both build the same tables, and finished steps are skipped.

In [ ]:
# 1a Libraries and the project's own data-loading code
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add analysis/ to Python's search path, so "from src..." imports work from inside notebooks/
PROJECT_ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "pl_data.py").exists())
sys.path.insert(0, str(PROJECT_ROOT))
from src.pl_data import DATA_DIR, FLAT_DIR, describe_structure, flatten_all, load_flat, load_raw

pd.set_option("display.max_rows", 200)   # long summary tables show in full
pd.set_option("display.max_colwidth", 140)

# The chart style every chart below uses (src/eda_helpers.py's use_chart_style(), written out here). The four
# categorical colours are in a fixed order that is never reordered, so a given group keeps the same colour in
# every chart and every notebook; ORANGE is reserved for the one group being highlighted.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, INK_2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#c9c8c2", "#e4e3df", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": INK_2, "ytick.color": INK_2, "text.color": INK,
    "font.size": 10.5, "lines.linewidth": 2, "legend.frameon": False, "figure.dpi": 110,
})

In [ ]:
# 1b Load the raw documents ({} when every table is already flattened)
raw = load_raw()

In [ ]:
# 1c Column names and the Python type of each value in the first row
describe_structure(raw)

In [ ]:
# 1d Flatten all four tables (skipped when already done)
flatten_all(raw)

In [ ]:
# 1e Build the full-file tables (skipped when done)
from src.stream_full import build

BUILD_MODE = "auto"  # "bulk": one parallel pass | "stream": three memory-light passes | "auto": bulk when the machine can take it
# Explicitly asking for "bulk" on a machine with under 16 GB RAM / 8 threads raises, rather than risk a crash
# or thrashing partway through a multi-hour run; pass build(BUILD_MODE, force=True) to override that check.
build(BUILD_MODE)

## 2. Data cleaning

Full-file fixes (13.4M listings), data quality only: nothing is removed but data errors. The cleaning report (2f) lists every action.

- **Text** becomes counts and flags in the stream (`src/listing_columns.py`). Without a description a text flag is unknown, not False.
- **Dates:** impossible timelines are removed; implausible `posted` dates are blanked, and the listing stays.
- **Reposts** are kept and flagged, with features built from earlier copies only. Duplicate ids are dropped.
- **Pay** becomes one hourly rate; reversed ranges are swapped, wrong units re-read, and values outside $10k–$1M a year blanked.
- **Derived columns:** `live` (last seen within 2 days of September 30, so still open then: its lifespan is censored), `days_open`, `company_earlier` and the career-change flags.
- **PredictLeads crawl artifacts** (batch first sightings, dropouts) are flagged and kept.

**Why listings without a description stay in scope:** about 17-18% of listings have none, and that share isn't
steady - it rises to 23-25% from June on, against 6-8% in March-April, which points at a PredictLeads collection change, not a
steady rate of employers skipping the field. These listings also close in a median of 3 days against 30 for
everything else - the same shape as a PL crawl artifact (section 2e), not obviously real behavior, and still
unverified (asked of PredictLeads directly, `PROJECT.md`). Missing pay looked predictive only because of this
group, a confound `EDA_SDA_2` already caught and corrected for. So `has_desc` is flagged, not dropped, the same
as every other text blocker in this project (years asked, degree required, clearance, ...): a label and an
opt-in filter, with the count of what it would hide always shown - not a silent cut at the data layer that would
also erase most of June onward before PredictLeads has explained why this happened.

**Not here yet, on purpose:** this section works on all 13.4M listings, before the scope cut, so a few fields the
app needs later - `recruiter_title`, `salary_text` (pay as written), `location_text`, `url`, `record_type`,
`company_record_type` - are not loaded or computed in this notebook at all. They are fetched only for the ~900K
in-scope listings, in section 4 (`rows_table.scope_fields()`, joined into the in-scope table there) - loading them for
every listing here would cost memory for the 93% that never reach the scope.

In [ ]:
# 2a-1 Where the cleaned full-file rows live, and which columns to load
from src.rows_table import label_onehot, load_rows, row_extras

SECTION_START = time.time()  # printed in 2f-3, once section 2's cleaning is done

CLEAN_DIR = DATA_DIR / "clean"
# This notebook writes to its own folder, not next to EDA_SDA_1's tables: the two produce the same table
# under the same name but with different columns (plain contract_* booleans here, a packed contract_bits
# there), and until this rewrite has been checked against the original they both have to exist to be
# compared. Section 7's hand-off to EDA_SDA_2 only moves to CLEAN_DIR once this notebook replaces EDA_SDA_1.
LEAN_DIR = CLEAN_DIR / "lean"
LEAN_DIR.mkdir(parents=True, exist_ok=True)

# Every column here is listing_columns.lean_part()'s output (src/listing_columns.py: COLS + the derived text flags),
# minus id/title - row_extras() below reads those two directly to build id_fp/repost_fp instead, so the raw text
# never needs to enter this notebook's memory.
#
# Two kinds of columns are deliberately NOT loaded here, for two different reasons:
#  - Columns that exist at this full-file scale but aren't needed by this notebook: job_title, has_recruiter,
#    salary_currency, salary_low/high (pre-USD-conversion), was_translated, last_processed - added by
#    src/stream_full.py's typed_rows(), reserved for EDA_SDA_2's feature screening.
#  - Columns that do not exist yet at all, at this (13.4M-row) scale: recruiter_title, salary_text (pay as
#    written), location_text, url, record_type, company_record_type. These get fetched only for the ~900K scope
#    listings, in SECTION 4 (4d: rows_table.scope_fields(), joined with final.join(extras)) - not here, and not
#    anywhere before then. They matter for the app (e.g. url -> the listing's "apply" link), which is exactly
#    why they are kept at all, just deferred: loading them here would cost memory for the 93% of listings that
#    never reach the scope.
LOAD_COLS = ["company_id", "first_seen", "last_seen", "posted", "status", "seniority", "language", "onet_code",
             "occupation_group", "occupation", "salary_low_usd", "salary_high_usd", "salary_unit", "has_desc",
             "has_html", "desc_words", "mentions_remote", "urgency", "years_required", "entry_wording",
             "changer_wording", "degree_required", "equivalent_experience", "n_locations", "country",
             "source_type", "platform"]

# The stream writes its row progress here; it must say "finished" before load_rows() can read every part
stream = json.loads((FLAT_DIR / "stream_rows_progress.json").read_text())
if not stream.get("finished"):
    raise RuntimeError("The rows stream hasn't finished: run `python -m src.stream_full --rows` first.")

In [ ]:
# 2a-2 Load the cleaned rows (13.4M listings, light columns only)
start = time.time()
jobs = load_rows(LOAD_COLS)
print(f"loaded {len(jobs):,} rows and {jobs.shape[1]} columns in {time.time() - start:.0f}s, "
      f"{jobs.memory_usage(deep=True).sum() / 1e9:.1f} GB in memory")

In [ ]:
# 2a-3 Each row's extra fields: id and repost fingerprints, counts, and a plain column per contract type and
# category (label_onehot: the Lean alternative to a bit-packed column, at the cost of one byte per label per row)
extras, labels = row_extras()
onehot = label_onehot(labels)
jobs = pd.concat([jobs, extras.drop(columns=["contract_bits", "category_bits"]), onehot], axis=1)
del extras, onehot

In [ ]:
# 2a-4 Each company's first country, for the comparisons in section 4
from src.listing_columns import first_country

companies = load_flat("companies", ["id", "attributes.location_data"])
companies["country"] = companies.pop("attributes.location_data").map(first_country)
companies = companies.drop_duplicates("id")

In [ ]:
# 2a-5 Start the cleaning report: one row per action, filled in through section 2
report = [{"step": "2a", "action": "rows in the cleaned rows table (every English or unlabeled, US or no-country listing)",
           "rows": len(jobs)}]

In [ ]:
# 2b Standardize category labels (trim and lowercase, one column's distinct labels at a time)
for col in ["status", "seniority", "language", "salary_unit"]:
    before = jobs[col].nunique()
    jobs[col] = jobs[col].map(lambda c: str(c).strip().lower() if pd.notna(c) else c).astype("category")
    report.append({"step": "2b", "action": f"{col}: distinct labels merged by trimming and lowercasing",
                   "rows": before - jobs[col].nunique()})

# contract_types labels were already unified part by part in 2a (row_extras); just record how many merged
report.append({"step": "2b", "action": "contract_types: distinct labels merged (underscores, case, spacing)",
               "rows": labels["raw_contract_labels"] - len(labels["contract_types"])})

In [ ]:
# 2c-1 Remove listings with impossible timelines
DATA_END = pd.Timestamp("2026-09-30", tz="UTC")
END_OF_DATA = DATA_END + pd.Timedelta(days=1)

rules = {
    "missing first_seen or last_seen": jobs.first_seen.isna() | jobs.last_seen.isna(),
    "last_seen before first_seen": jobs.last_seen < jobs.first_seen,
    "dates after September 30 (the data's end date)": (jobs.first_seen > END_OF_DATA) | (jobs.last_seen > END_OF_DATA),
}
bad = pd.Series(False, index=jobs.index)
for name, mask in rules.items():
    report.append({"step": "2c", "action": f"removed, {name}", "rows": int((mask & ~bad).sum())})
    bad |= mask
jobs = jobs[~bad]
del rules

In [ ]:
# 2c-2 Blank implausible employer-posted dates (the listing itself stays)
bad_posted = (jobs.posted > jobs.first_seen + pd.Timedelta(days=1)) | (jobs.posted < pd.Timestamp("2000-01-01", tz="UTC"))
jobs.loc[bad_posted, "posted"] = pd.NaT
report.append({"step": "2c", "action": "posted blanked (after first_seen or before 2000)", "rows": int(bad_posted.sum())})

In [ ]:
# 2c-3 Drop duplicate listing ids (two different ids sharing a fingerprint is about 1-in-200,000 across 13M listings)
dup_id = jobs.id_fp.duplicated()
report.append({"step": "2c", "action": "removed, duplicate id", "rows": int(dup_id.sum())})
jobs = jobs[~dup_id].drop(columns="id_fp")

In [ ]:
# 2c-4 Flag reposts (same company, title and description as an earlier listing); kept, not dropped
jobs = jobs.sort_values("first_seen")
has_key = jobs.repost_fp.notna()
jobs["is_repost"] = has_key & jobs.duplicated("repost_fp")
report.append({"step": "2c", "action": "flagged as repost (kept, excluded from the EDA tables)",
               "rows": int(jobs.is_repost.sum())})
report.append({"step": "2c", "action": "postings without a description, can't be checked for reposts",
               "rows": int((~has_key).sum())})

In [ ]:
# 2c-5 Repost features: built only from earlier copies of the same job, so they're known when a listing appears
# (the table is sorted by first_seen from 2c-4, so a job's rows come in listing order within each group)
same_job = jobs[has_key].groupby("repost_fp")

jobs["times_listed"] = np.float32(np.nan)  # unknown without a description
jobs.loc[has_key, "times_listed"] = (same_job.cumcount() + 1).astype("float32")  # 1 for the first listing, 2 for the next, ...

jobs["days_since_first_listed"] = np.float32(np.nan)  # since the job's original listing
jobs.loc[has_key, "days_since_first_listed"] = (
    jobs.loc[has_key, "first_seen"] - same_job.first_seen.transform("min")).dt.total_seconds().div(86400).astype("float32")

jobs["days_since_repost_last_listed"] = np.float32(np.nan)  # since the job's previous listing; 0 for the first listing
jobs.loc[has_key, "days_since_repost_last_listed"] = (
    jobs.loc[has_key, "first_seen"] - same_job.first_seen.shift()).dt.total_seconds().fillna(0).div(86400).astype("float32")
del same_job

In [ ]:
# 2d-1 Fix reversed salary ranges (low above high) and take the midpoint
reversed_range = jobs.salary_low_usd > jobs.salary_high_usd
jobs.loc[reversed_range, ["salary_low_usd", "salary_high_usd"]] = (
    jobs.loc[reversed_range, ["salary_high_usd", "salary_low_usd"]].to_numpy())
mid = jobs[["salary_low_usd", "salary_high_usd"]].mean(axis=1)
report.append({"step": "2d", "action": "salary present (midpoint of low and high)", "rows": int(mid.notna().sum())})
report.append({"step": "2d", "action": "salary range reversed (low above high): swapped", "rows": int(reversed_range.sum())})

In [ ]:
# 2d-2 Re-read salary units that are clearly mislabeled ("$25 a year" is really hourly, and the reverse)
PER_YEAR = {"hour": 2080, "day": 260, "week": 52, "month": 12, "year": 1}


def unit_factor(u):
    return next((f for k, f in PER_YEAR.items() if k in u), np.nan) if pd.notna(u) else np.nan


yearly_but_hourly = jobs.salary_unit.str.contains("year", na=False) & mid.between(7, 200)
hourly_but_yearly = jobs.salary_unit.str.contains("hour", na=False) & mid.between(10_000, 1_000_000)
# .map() applies unit_factor once per distinct label rather than once per row (that part needs no helper).
# But when every label maps to a different number - as these five units do - pandas keeps the result
# categorical, and a categorical can't be multiplied in 2d-3 (TypeError: cannot perform the numpy op
# multiply). .astype("float64") turns the labels back into plain numbers; checked against the helper this
# replaced, src.rows_table.cat_map, on both a one-to-one and a many-to-one mapping (tests/test_lean_cat_map.py).
factor = (jobs.salary_unit.map(unit_factor).astype("float64")
          .mask(yearly_but_hourly, PER_YEAR["hour"]).mask(hourly_but_yearly, PER_YEAR["year"]))
report.append({"step": "2d", "action": "labelled yearly but clearly hourly ($7-$200): re-read as hourly",
               "rows": int(yearly_but_hourly.sum())})
report.append({"step": "2d", "action": "labelled hourly but clearly yearly ($10k-$1M): re-read as yearly",
               "rows": int(hourly_but_yearly.sum())})
report.append({"step": "2d", "action": "salary present but unit missing or not recognized",
               "rows": int((mid.notna() & factor.isna()).sum())})

In [ ]:
# 2d-3 Convert to one hourly rate and blank pay outside a plausible range ($10k-$1M a year)
HOURS_PER_YEAR = PER_YEAR["hour"]
hourly = mid * factor / HOURS_PER_YEAR
outlier = ~hourly.between(10_000 / HOURS_PER_YEAR, 1_000_000 / HOURS_PER_YEAR) & hourly.notna()
jobs["salary_hourly_usd"] = hourly.mask(outlier).astype("float32")
jobs["pay_suspect"] = reversed_range | yearly_but_hourly | hourly_but_yearly
report.append({"step": "2d", "action": "hourly pay outside $4.81-$480.77 ($10k-$1M a year), blanked", "rows": int(outlier.sum())})
report.append({"step": "2d", "action": "usable hourly pay", "rows": int(jobs.salary_hourly_usd.notna().sum())})

In [ ]:
# 2e-1 Derived timing columns: still open, days open, and the first-seen month
LIVE_DAYS = 2
jobs["live"] = (DATA_END - jobs.last_seen).dt.days <= LIVE_DAYS
jobs["days_open"] = ((jobs.last_seen - jobs.first_seen).dt.total_seconds() / 86400).astype("float32")
jobs["month"] = jobs.first_seen.dt.tz_localize(None).dt.to_period("M")
jobs["occupation_group"] = jobs.occupation_group.cat.add_categories(["(missing)"]).fillna("(missing)")

In [ ]:
# 2e-2 Career-change accessibility flags (see the decisions in section 2)
ENTRY_LEVELS = {"junior"}  # the only entry-level label (the others: mid_senior, manager, head, director,
                           # vice_president, president, c_level, partner, founder, not_set)
jobs["entry_seniority"] = jobs.seniority.isin(ENTRY_LEVELS)
jobs["low_experience_ask"] = jobs.years_required <= 2  # False where no years are stated
jobs["career_open"] = jobs.entry_seniority | jobs.low_experience_ask | jobs.entry_wording | jobs.changer_wording

In [ ]:
# 2e-3 Company history in any field, counted before the cut (a confounder for every later notebook): each
# listing's count of its company's listings first seen before it (same-day listings count together; reposts included)
jobs["company_earlier"] = (jobs.groupby("company_id", observed=True).first_seen.rank(method="min") - 1).astype("float32")

### PredictLeads crawl artifacts: when the date is the PredictLeads crawler's, not the listing's

PredictLeads doesn't watch job postings directly - it periodically re-crawls each company's job pages and
records what it currently sees there. `first_seen` is the first time the PL crawler happened to notice a listing,
not necessarily when it was actually posted; `last_seen` is the last time the PL crawler saw it, not necessarily
when it closed. Almost always these track reality closely. Two patterns don't, and both are caused by the
PL crawler, not by hiring:

- **`batch_first_seen`:** PredictLeads starts crawling a company for the first time, or the company rebuilds its
  careers page and gets re-indexed - and dozens of jobs that already existed all get a `first_seen` of that one
  day. It looks like a hiring surge; it's really the PL crawler catching up.
- **`crawl_dropout`:** PredictLeads loses track of a company's page (a PL crawler bug, a site change it can't
  parse), and a batch of that company's listings all get a `last_seen` of that one day. It looks like they all
  closed together; `days_open` for those listings may be cut artificially short.

**The one-day batch rule** detects both the same way: a company with at least 5 listings, where 60% or more of
them share one exact day for the date in question (`first_seen` for batches, `last_seen` for dropouts), gets
flagged. Those two numbers (5, 60%) are a judgment call, not derived from anything - meant to catch an obvious
pile-up, not flag every small coincidence. The data's first two days are excluded from batch detection entirely:
at the very start of the PL crawl, *every* company's existing backlog gets dated to day one, which would make every
company look batched - that's a dataset-wide startup effect, not the per-company signal this rule is for.

Both get **flagged, kept, never dropped** - they're still real listings, just ones where a timing-based measure
(how many jobs a company posted on one day, or how long a listing stayed open) might read oddly for that
company.

**Return to this later (queued in `ANALYSIS.md`, after the 2026-10-29 pitch):** checked on synthetic data, the
fixed 5-listings/60%-share rule has a real blind spot - it misses genuine one-day clustering entirely below 60%
concentration, where a company-size-adjusted (shuffled-null) significance test would catch it reliably. Separately,
treating `crawl_dropout` listings as a real close (instead of censored) measurably biases the Kaplan-Meier curve
short - modest at the project's actual dropout rate (~4% on median days-open), but real. Both are evidenced, not
just suspected; neither is fixed here because it would change real `EDA_SDA_1`/`EDA_SDA_2` numbers during a
scope freeze.

In [ ]:
# 2e-4 The one-day batch rule (see the markdown above): flags listings on a day holding 60%+ of their
# company's eligible listings (5+ of them)
BATCH_MIN, BATCH_SHARE = 5, 0.6


def one_day_batch(company_codes, day, eligible):
    """True for listings on the day holding BATCH_SHARE+ of their company's eligible listings (BATCH_MIN+ of them)."""
    d = pd.DataFrame({"co": company_codes[eligible], "day": day[eligible]})
    d = d[d.co >= 0]
    per_day = d.groupby(["co", "day"]).size()
    per_co = per_day.groupby(level="co").sum()
    top = per_day.sort_values(ascending=False).groupby(level="co").head(1)  # the busiest day per company
    top = top[(per_co.reindex(top.index.get_level_values("co")).to_numpy() >= BATCH_MIN)
              & (top.to_numpy() / per_co.reindex(top.index.get_level_values("co")).to_numpy() >= BATCH_SHARE)]
    hit = pd.MultiIndex.from_arrays([company_codes, day]).isin(top.index)
    return pd.Series(hit & eligible, index=jobs.index)

In [ ]:
# 2e-5 Apply the batch and dropout rules
codes = jobs.company_id.cat.codes.to_numpy()
first_day = (jobs.first_seen.astype("int64") // 86_400_000_000_000).to_numpy()
last_day = (jobs.last_seen.astype("int64") // 86_400_000_000_000).to_numpy()
jobs["batch_first_seen"] = one_day_batch(codes, first_day, (jobs.first_seen >= pd.Timestamp("2026-04-01", tz="UTC")).to_numpy())
jobs["crawl_dropout"] = one_day_batch(codes, last_day, (~jobs.live).to_numpy())
del codes, first_day, last_day

In [ ]:
# 2e-6 How much the PL crawl artifacts affect the scope, in each company's own terms
BIG_COMPANIES = (jobs.company_id.value_counts() >= BATCH_MIN).sum()
artifact_stats = {
    "listings first seen in a company's one-day batch": jobs.batch_first_seen.mean(),
    "companies with such a batch (of companies with 5+ listings)":
        jobs.loc[jobs.batch_first_seen, "company_id"].nunique() / max(BIG_COMPANIES, 1),
    "listings that disappeared in a company's one-day dropout": jobs.crawl_dropout.mean(),
    "companies with such a dropout (of companies with 5+ listings)":
        jobs.loc[jobs.crawl_dropout, "company_id"].nunique() / max(BIG_COMPANIES, 1),
}

In [ ]:
# 2e-7 Save every cleaned listing to disk, then read back the analysis table (originals only, reposts excluded)
CLEANED_PATH = LEAN_DIR / "cleaned_rows_full.parquet"
jobs.to_parquet(CLEANED_PATH)
JOBS_COLS = ["company_id", "first_seen", "occupation_group", "onet_code", "is_repost", "language", "country",
             "has_desc", "mentions_remote"]
jobs = jobs[JOBS_COLS + [c for c in jobs.columns if c.startswith("contract_")]]
jo = pd.read_parquet(CLEANED_PATH, filters=[("is_repost", "==", False)])
report.append({"step": "2e", "action": "postings in the analysis table (originals only)", "rows": len(jo)})

In [ ]:
# 2f-1 The cleaning report: every action, in order, with its share of the starting count
from IPython.display import display

cleaning_report = pd.DataFrame(report)
cleaning_report["share of start"] = (cleaning_report.rows / report[0]["rows"]).round(4)
print("Cleaning report (every action, in order)")
display(cleaning_report)

In [ ]:
# 2f-2 The analysis table at a glance (originals only: the rows for the every-field comparison)
print("Analysis table at a glance (originals: the rows for the every-field comparison)")
display(pd.Series({
    "postings": f"{len(jo):,}",
    "companies": f"{jo.company_id.nunique():,}",
    "occupation groups": jo.occupation_group.nunique(),
    "still open on September 30": f"{jo.live.mean():.1%}",
    "first_seen from": jo.first_seen.min().date(),
    "first_seen to": jo.first_seen.max().date(),
}).to_frame("value"))

In [ ]:
# 2f-3 The PL crawl artifacts, and how long section 2 took
print("PL crawl artifacts (flagged, kept): every cleaned listing")
display(pd.Series(artifact_stats).to_frame("share"))
print(f"section 2 done in {(time.time() - SECTION_START) / 60:.1f} min")

**What the data shows**

- **Cleaning removed no listings:** no duplicate ids, no impossible timelines, no reversed pay ranges and no
mislabelled pay units - every one of the 5,625,697 listings that state pay states a unit the converter
recognizes. It flagged 1,366,992 reposts (10.5%; a further 2,223,437 listings, 17.1%, have no description and so
can't be checked at all), blanked 349,499 employer posting dates (2.7%: placeholders before 2000, or dates after
the PredictLeads crawler first saw the listing) and blanked 984 pay values outside $10K-$1M a year.
- **11,650,472 original listings from 323,665 companies** feed the every-field comparison in section 3; 22.3% of
them were still open on September 30, which is why section 3 needs Kaplan-Meier rather than a plain average.
- **PredictLeads crawl artifacts are small but concentrated:** 2.4% of listings were first seen in a company's one-day batch
(16.8% of the companies with 5+ listings have one) and 1.2% disappeared in a one-day dropout (9.9% of those
companies). They distort single companies far more than the totals, so they are flagged, never dropped.
- **The same numbers as `EDA_SDA_1`,** action by action: the original prints them rounded (13.0M, 1.4M, 349K,
11.7M, 324K) and reports the same 984 blanked pay values. Section 2 takes 5.2 minutes here.

## 3. Q1: How do computer and mathematical roles compare with every other field?

Every occupation group before the scope cut, on original English listings; weekly rates use full weeks only.
**Context, not cause:** fields differ in company size, source and seniority.

### Why this needs Kaplan-Meier

**22.3% of all original listings were still open on September 30** - not a tail case, roughly a quarter of the
data - and it isn't spread evenly: a listing posted in late September has barely had time to close, so it's far
more likely to still be open than one posted in April. Any "typical time to close" or "closed within 7 days"
computed only from listings that have already closed would read short, worst for the most recently posted
listings in every field. `median days open` and `closed within N days` below use the Kaplan-Meier curve
(`src/kaplan_meier.py`, `docs/kaplan_meier.md`) for exactly this reason, computed within each occupation group
from its closed *and* still-open listings together - not just the ones that happen to still be open.

This isn't a one-off, either: `TimeSeries` runs Kaplan-Meier by month for the same reason (each month's cohort
has a different amount of follow-up); `EDA_SDA_4` fits a Cox model; `ML_Rank`'s priority queue trains an actual
survival forest (C-index 0.725) for its "chance still open" factor. None of those are possible without a method
that uses the closed and still-open listings together.

In [ ]:
# 3a Shared constants, and which weeks count as full weeks
COMPANY = "company_id"
SCOPE_GROUP = "Computer and Mathematical"  # the field this project is about; highlighted in every comparison below

# Weekly rates use full weeks only: the first week holds the PL crawler's start-up backlog (every listing already up
# when it began is first seen then), and the last week is cut off by the data's end date (Wednesday, September 30).
# The listings themselves stay in every table; only the per-week rates leave those two weeks out.
week = jo.first_seen.dt.tz_localize(None).dt.to_period("W")
FIRST_WEEK, LAST_WEEK = week.min(), week.max()
jo["full_week"] = (week > FIRST_WEEK) & (week < LAST_WEEK)
FULL_WEEKS = week[jo.full_week].nunique()
del week

# Section 4's language rule, applied here already so the fields are compared on the same footing as the scope
in_market = jo.language.eq("en").to_numpy()

print(f"full weeks: {FULL_WEEKS} (first week {FIRST_WEEK}, last week {LAST_WEEK} left out of weekly rates)")
print(f"English original listings: {in_market.sum():,} of {len(jo):,}")

In [ ]:
# 3b Every field side by side (original listings, English; text measures among listings with a description)
from src import kaplan_meier as km  # the survival method the markdown above explains (docs/kaplan_meier.md)

FIELD_COLS = ["occupation_group", COMPANY, "days_open", "live", "salary_hourly_usd", "entry_seniority",
              "career_open", "years_required", "degree_required", "mentions_remote", "has_desc", "full_week"]
market = jo.loc[in_market, FIELD_COLS]


def field_stats(g):
    """One row of the comparison table: every measure for one occupation group.

    Two measures need care. "median days open" and "closed within 7 days" come from the Kaplan-Meier curve, not
    from the listings that happen to have closed, for the reason given in the markdown above - a listing is
    "finished" when it is no longer live (~g.live). And the pay median is only taken where some pay is actually
    disclosed: a field with none would otherwise make numpy warn about the median of an empty selection, and the
    answer we want in that case is simply "no pay disclosed" (NaN).
    """
    curve = km.km_curve(g.days_open, ~g.live)
    described = g[g.has_desc]          # the text flags only mean anything where there is a description to read
    stated_years = g.years_required.dropna()
    return pd.Series({
        "listings": len(g),
        "share of all listings": len(g) / len(market),
        "new per week": g.full_week.sum() / FULL_WEEKS,
        "companies": g[COMPANY].nunique(),
        "median hourly pay (USD)": g.salary_hourly_usd.median() if g.salary_hourly_usd.notna().any() else np.nan,
        "pay disclosed": g.salary_hourly_usd.notna().mean(),
        "median days open (Kaplan-Meier)": km.median_survival(g.days_open, ~g.live),
        "closed within 7 days": 1 - km.survival_at(curve, 7),
        "entry-level seniority": g.entry_seniority.mean(),
        "asks 2 years or fewer (where stated)": (stated_years <= 2).mean(),
        "open to career changers (any sign)": g.career_open.mean(),
        "degree required": described.degree_required.mean(),
        "mentions remote": described.mentions_remote.mean(),
    })


fields = market.groupby("occupation_group", observed=True).apply(field_stats, include_groups=False)
fields = fields.drop(index="(missing)", errors="ignore").sort_values("listings", ascending=False)

# One format per measure, written out, rather than guessed from the column's name. Shares get one decimal
# throughout (the shared helper switched between 0, 1 and 2 decimals by size), so 9.4% and 3.2% read the same
# way as 76.0% - these are the numbers ../DIFFERENTIATORS.md quotes.
FIELD_FORMATS = {
    "listings": "{:,.0f}", "share of all listings": "{:.1%}", "new per week": "{:,.0f}",
    "companies": "{:,.0f}", "median hourly pay (USD)": "${:,.2f}", "pay disclosed": "{:.1%}",
    "median days open (Kaplan-Meier)": "{:,.1f}", "closed within 7 days": "{:.1%}",
    "entry-level seniority": "{:.1%}", "asks 2 years or fewer (where stated)": "{:.1%}",
    "open to career changers (any sign)": "{:.1%}", "degree required": "{:.1%}", "mentions remote": "{:.1%}",
}


def highlight_scope(row):
    """Outline one row in orange, so this project's field is findable in a 23-row table."""
    style = f"font-weight: bold; border-top: 2px solid {ORANGE}; border-bottom: 2px solid {ORANGE}"
    return [style if row.name == SCOPE_GROUP else "" for _ in row]


# Each column is shaded on its own scale (a share of 9% and a pay of $66 can't share one scale). Columns with
# nothing to compare - all missing, or every field the same - are left unshaded instead of coming out solid.
styled = fields.style.format(FIELD_FORMATS).apply(highlight_scope, axis=1)
for c in fields.columns:
    if fields[c].notna().sum() > 1 and fields[c].nunique() > 1:
        styled = styled.background_gradient(cmap="Blues", subset=[c], low=0.05, high=0.35)

# The printed label stays "3a." : it is what ../DIFFERENTIATORS.md and ../PROJECT.md cite as "EDA_SDA_1 3a".
# Renumbering it to match this cell's own 3b would silently break those citations.
print(f"\n3a. Every field side by side ({len(fields)} occupation groups; original English listings; "
      f"each column shaded on its own scale)")
display(styled)

In [ ]:
# 3b (cont.) Where computer and mathematical stands among the fields
standing = pd.DataFrame({
    "computer and mathematical": fields.loc[SCOPE_GROUP],
    "median field": fields.median(),
    "rank (1 = highest)": fields.rank(ascending=False, method="min").loc[SCOPE_GROUP].astype(int),
})

# Here the measures are the rows, so each row carries its own unit and has to be formatted on its own (the two
# value columns hold pay, counts and shares all at once). astype(object) first, so writing a formatted string
# into a numeric column doesn't fight the column's dtype.
shown = standing.astype(object)
for measure, fmt in FIELD_FORMATS.items():
    for col in ["computer and mathematical", "median field"]:
        value = standing.loc[measure, col]
        shown.loc[measure, col] = fmt.format(value) if pd.notna(value) else ""

print(f"\n3a. Where computer and mathematical stands among the {len(fields)} fields")
display(shown)

In [ ]:
# 3c chart: openness to career changers vs. pay, every field (bubble size = listings; computer and mathematical
# highlighted). The two dashed lines are all listings' own figures, so each field reads as above or below them.
x = 100 * fields["open to career changers (any sign)"]
y = fields["median hourly pay (USD)"]

fig, ax = plt.subplots(figsize=(10, 5.6))
ax.scatter(x, y, s=40 + 900 * fields.listings / fields.listings.max(), alpha=0.8, edgecolor=SURFACE,
           color=[ORANGE if name == SCOPE_GROUP else BLUE for name in fields.index])

# Labelling all 23 fields would overlap into noise: the 12 biggest, plus this project's field wherever it lands.
for name in fields.index[:12].union([SCOPE_GROUP]).intersection(fields.index):
    ax.annotate(str(name)[:30], (x[name], y[name]), xytext=(6, 4), textcoords="offset points", fontsize=8,
                color=INK if name == SCOPE_GROUP else INK_2,
                weight="bold" if name == SCOPE_GROUP else "normal")

ax.axvline(100 * market.career_open.mean(), color=MUTED, ls="--", lw=1)
ax.axhline(market.salary_hourly_usd.median(), color=MUTED, ls="--", lw=1)
ax.set(title="Open to career changers vs. pay, every field (dashed: all listings; orange: computer and mathematical)",
       xlabel="% of listings with any sign of being open to career changers", ylabel="median hourly pay ($)")
plt.tight_layout()
plt.show()

**Saved for later notebooks:** the occupation summary (pay, timing and openness per occupation; the compass's "your pay increase"), `occupation_summary.pkl`, and company activity (each company's listings in any field and newest listing; the "company still posting" ghost signal), `company_activity.parquet`. Aggregates only; nothing is shown. Like the cleaned rows, both are written to `data/clean/lean/` until this notebook replaces `EDA_SDA_1`.

In [ ]:
# 3d Occupation summary (every field; aggregates only, not shown) and company activity (every field)
def occupation_stats(g):
    """One row of the occupation summary: the measures later notebooks read, for one occupation."""
    return pd.Series({
        "listings": len(g),
        "companies": g[COMPANY].nunique(),
        "pay disclosed": g.salary_hourly_usd.notna().mean(),
        "median hourly pay (USD)": g.salary_hourly_usd.median() if g.salary_hourly_usd.notna().any() else np.nan,
        "median days open (Kaplan-Meier)": km.median_survival(g.days_open, ~g.live),
        "entry-level seniority": g.entry_seniority.mean(),
        "open to career changers (any sign)": g.career_open.mean(),
    })


OCCUPATION_COLS = ["occupation_group", "occupation", COMPANY, "salary_hourly_usd", "days_open", "live",
                   "entry_seniority", "career_open"]
occupation_summary = (jo.loc[in_market, OCCUPATION_COLS]
                      .groupby(["occupation_group", "occupation"], observed=True)
                      .apply(occupation_stats, include_groups=False))
OCC_SUMMARY_PATH = LEAN_DIR / "occupation_summary.pkl"
occupation_summary.to_pickle(OCC_SUMMARY_PATH)

# Company activity counts a company's listings in every field, reposts included, so it reads `jobs` (every cleaned
# listing), not `jo` (originals only). The "company still posting" ghost signal needs the newest listing of any kind.
company_activity = jobs.groupby(COMPANY, observed=True).agg(listings_any_field=("first_seen", "size"),
                                                            newest_listing_any_field=("first_seen", "max"))
COMPANY_ACTIVITY_PATH = LEAN_DIR / "company_activity.parquet"
company_activity.to_parquet(COMPANY_ACTIVITY_PATH)

del market
print(f"saved {len(occupation_summary):,} occupations in {occupation_summary.index.get_level_values(0).nunique()} groups "
      f"({OCC_SUMMARY_PATH.name}) and {len(company_activity):,} companies ({COMPANY_ACTIVITY_PATH.name})")

## 4. The scope: where the data narrows to the roles TripleTen trains for

Before the cut, every field; after it, the scope only. Nothing is deleted: the cleaned rows stay on disk.

| Decision | Choice | Why |
|:--|:--|:--|
| **Language** | English; listings with no language label dropped | The text features and the app are English; unlabeled listings have no description (4b) |
| **Country** | Listings that name the US; no-country listings dropped | Pay is in dollars and the job-growth projections (BLS) are for the US; only these are known US openings (4b) |
| **Scope** | The roles TripleTen trains for, plus adjacent tech roles; credentialed math out | TripleTen's program catalog, a neutral definition (4a) |
| **Not a tech path** | Three occupations dropped: document management specialists (15-1299.03), GIS technologists (15-1299.02) and search marketing strategists (13-1161.01) | Records work, mapping and search marketing aren't a tech path a career changer would take. Each is found by an exact O*NET code, so the rule can't misfire, and leaving them in would only add noise to every later table (4c, 4c-4) |
| **Description** | Listings with no description are kept and flagged `has_desc`, not dropped | 17-18% have none, rising from 6-8% in March-April to 23-25% from June (which points at a PredictLeads collection change, not employers changing, and is still unverified), and they close in a median 3 days vs. 30. Dropping them would erase most of June onward and hide a confounder: missing pay looks predictive only through this group. For model fitting, every text feature is blank (not False) without a description, so `has_desc` has to sit next to them for a model to tell "no description" from "a description that doesn't mention it" (section 2; `EDA_SDA_2` 3.9, `EDA_SDA_4`, `ML_Rank`) |
| **Time** | April–September; late March kept, flagged `partial_month` | Left out of per-month counts only |

Only whole occupations that aren't a tech path are dropped here. Seniority, long-open listings and staffing agencies are tagged or hidden in `EDA_SDA_2`, never dropped.

### 4a Why this scope

- **TripleTen's program catalog** (tripleten.com, checked 2026-10-10) defines it: a neutral, outside definition. IT support, networks and systems, database and IT project management have no program but stay as **adjacent tech roles,** since IT support is the most open entry point.
- **The data supports it:** a switch pays off (section 3), 2,813 career changers in the résumé data moved into these roles (`Profiles.ipynb`), and they're hard to enter, which is where hiding and ranking help.
- **What is left out:** credentialed math (actuaries and mathematicians need a credential) and three occupations that aren't a tech path, document management specialists (15-1299.03), GIS technologists (15-1299.02) and search marketing strategists (13-1161.01). The test for an occupation: would a TripleTen graduate, or someone moving up from IT support or networking, plausibly apply? The full list of occupations in the scope is 4c-4.

### 4b Language and country

What the two rules cost: no-country listings may be US remote jobs, and unlabeled listings may be the ones with no description.

In [ ]:
# 4b-1 Remote mentions, by whether the listing names a country (every cleaned listing, every field)
# A listing has a "remote contract type" if any of its contract labels says remote ("fully remote", "100% remote", ...).
# After the one-hot step in 2a-3 each label is its own True/False column, so that is a plain any() across those columns.
remote_columns = [c for c in jobs.columns if c.startswith("contract_") and "remote" in c]
remote_contract = jobs[remote_columns].any(axis=1)

by_country_known = pd.DataFrame({
    "country_known": jobs.country.notna(),
    "mentions_remote": jobs.mentions_remote,
    "remote_contract_type": remote_contract,
}).groupby("country_known").agg(
    listings=("mentions_remote", "size"),
    mentions_remote=("mentions_remote", "mean"),
    remote_contract_type=("remote_contract_type", "mean"),
)
print("4b. Remote mentions, by whether the country is known")
display(by_country_known)

In [ ]:
# 4b-2 Listings with no country: where are their companies based?
# Look up each listing's company in the companies table (2a-4). A company can be in the US, somewhere else, or
# unknown (no location, or not in the table); the last two are dropped along with the no-country listings.
company_country = jobs.company_id.astype(object).map(companies.set_index("id").country)
no_country = jobs.country.isna()
company_in_us = company_country.eq("United States")
company_known = company_country.notna()

print("4b. Listings with no country: where their companies are based (share of no-country listings)")
display(pd.Series({
    "company based in the US": (no_country & company_in_us).sum() / no_country.sum(),
    "company based elsewhere (dropped)": (no_country & company_known & ~company_in_us).sum() / no_country.sum(),
    "company location unknown or not in the companies table (dropped)": (no_country & ~company_known).sum() / no_country.sum(),
}).to_frame("share of no-country listings"))

In [ ]:
# 4b-3 Language label vs. description, and a check that no non-US listing is left after the stream
print("4b. Language label vs. description (listings)")
display(pd.crosstab(jobs.language.notna().map({True: "language known", False: "no language label"}),
                    jobs.has_desc.map({True: "has description", False: "no description"}), margins=True))
other_country = (jobs.country.notna() & jobs.country.ne("United States")).mean()
print(f"listings with a country other than the US after the stream: {other_country:.2%}")
del remote_contract, company_country, no_country, company_in_us, company_known

**What the data shows**

- **Listings with no country (8.0% of cleaned listings) are largely US remote jobs:** 30% mention remote work vs. 13% with a country, and 12.3% use a remote contract label vs. 7.8%. 93% come from a US-based company, 2.6% from a company elsewhere and 4.3% from an unknown one; all are dropped.
- **No language label means no description:** 623K of the 624K unlabeled listings have none, so dropping them also removes about 28% of all listings without a description (623K of 2.2M). 95% of cleaned listings are English, and no listing outside the US remains after the stream.
- **A correction to `EDA_SDA_1`:** it reports the remote contract label at 0.83% vs. 0.51%. That came from the packed `contract_bits` column, which holds only the first 63 labels *alphabetically* of 161, and "remote" falls after them. The plain one-hot columns here keep every label, so the figures above replace those.

### 4c The scope cut

In the scope: occupation group *Computer and Mathematical* or an O*NET code starting 15-, or one of two TripleTen roles outside the group (sales engineers, market research analysts); minus credentialed math (actuaries, mathematicians) and three occupations that aren't a tech path (document management specialists (15-1299.03), GIS technologists (15-1299.02) and search marketing strategists (13-1161.01)). Each listing gets a **role bucket** and a **TripleTen program** (`src/career_switch.py`), or "adjacent tech role" where no program trains for it.

In [ ]:
# 4c-1 The scope filter: roles TripleTen trains for, plus adjacent tech roles
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
from src.career_switch import EXTRA_ORDER, PROGRAM_ORDER, role_bucket, tripleten_program

FILTERS = {"language": "en", "country": "United States", "first_month": "2026-04"}
english = jobs.language.eq(FILTERS["language"])
english_us = english & jobs.country.eq(FILTERS["country"])

# Three ways into the scope. All three .map()/.str calls below run once per distinct label (a categorical column's
# trick), not once per row, so they are fast on 11.6M listings. na_action="ignore" leaves a missing code missing
# instead of passing it to role_bucket(), which expects a code string.
by_group = jobs.occupation_group.eq(SCOPE_GROUP)
by_code = jobs.onet_code.str.contains(r"^15-", na=False)           # O*NET codes starting 15- are computer and mathematical
bucket = jobs.onet_code.map(role_bucket, na_action="ignore")        # role bucket from the O*NET code (missing outside the mapped codes)
added = bucket.isin(EXTRA_ORDER)            # the two TripleTen roles outside the group (sales engineers, market research analysts)
credentialed = bucket.eq("credentialed math")  # actuaries and mathematicians: set aside
# Occupations that aren't a tech path, dropped by exact O*NET code (the test: would a TripleTen graduate, or someone moving up
# from IT support or networking, plausibly apply?). The full list of occupations in the scope is 4c-4.
NOT_A_TECH_PATH = {"15-1299.03": "document management specialists", "15-1299.02": "GIS technologists",
                   "13-1161.01": "search marketing strategists"}
not_tech_path = jobs.onet_code.str[:10].isin(NOT_A_TECH_PATH)
in_scope = english_us & (by_group | by_code | added) & ~credentialed & ~not_tech_path

print("4c. How English US listings enter the scope (share of English US listings)")
display(pd.Series({
    "occupation group is computer and mathematical": (english_us & by_group).mean() / english_us.mean(),
    "  of which the O*NET code also starts with 15-": (english_us & by_group & by_code).mean() / english_us.mean(),
    "only the O*NET code starts with 15- (group missing or different)": (english_us & by_code & ~by_group).mean() / english_us.mean(),
    "credentialed math (actuaries, mathematicians): set aside": (english_us & (by_group | by_code) & credentialed).mean() / english_us.mean(),
    "not a tech path (document management, GIS, search marketing): dropped":
        (english_us & (by_group | by_code | added) & not_tech_path).mean() / english_us.mean(),
    "a TripleTen role outside the group (sales engineers, business and market research analysts)":
        (english_us & added & ~(by_group | by_code)).mean() / english_us.mean(),
}).to_frame("share of English US listings"))

In [ ]:
# 4c-2 The funnel's steps, then read back only the scope's rows from disk
steps = [("listings in the full file", stream["records"]),
         ("English or unlabeled, US or no country (the stream, 2a)", stream["rows"]),
         ("cleaned: impossible dates and duplicate ids removed (2c)", len(jobs)),
         ("English only", english.sum()),
         ("United States only: the listing names the US", english_us.sum()),
         ("TripleTen roles plus adjacent tech roles (computer and mathematical, or an added TripleTen role)",
          (english_us & (by_group | by_code | added)).sum()),
         ("credentialed math out (actuaries, mathematicians)", (english_us & (by_group | by_code | added) & ~credentialed).sum()),
         ("occupations that aren't a tech path out (document management, GIS, search marketing)", in_scope.sum())]
scope_rows = jobs.index[in_scope.to_numpy()].to_numpy()  # the scope's row positions in the cleaned file

# Only the scope's rows come back from disk (every column); the every-field tables are no longer needed
del jo, jobs, english, english_us, by_group, by_code, bucket, added, credentialed, not_tech_path, in_scope, in_market
final = ds.dataset(CLEANED_PATH).to_table(filter=pc.field("__index_level_0__").isin(pa.array(scope_rows))).to_pandas()

final["partial_month"] = final.month < pd.Period(FILTERS["first_month"], "M")  # late March: flagged (hidden from per-month counts), never removed
week = final.first_seen.dt.tz_localize(None).dt.to_period("W")
final["full_week"] = (week > FIRST_WEEK) & (week < LAST_WEEK)

# The role bucket and the TripleTen program come from the O*NET code. A code that maps to nothing gets a visible label.
# .astype(object) first: .map() on a categorical column can return a categorical, which can't take a new label in fillna().
final["role_bucket"] = final.onet_code.map(role_bucket, na_action="ignore").astype(object).fillna("(unmapped 15- code)")
final["tripleten_program"] = final.onet_code.map(tripleten_program, na_action="ignore").astype(object).fillna("adjacent tech role")
steps.append((f"first seen before {FILTERS['first_month']}: kept, flagged partial_month ({final.partial_month.mean():.1%})", len(final)))
print(f"scope read back from disk: {len(final):,} listings")

In [ ]:
# 4c-3 The funnel table, and the scope by role bucket and TripleTen program
funnel = pd.DataFrame(steps, columns=["step", "listings"]).set_index("step")
funnel["removed"] = (-funnel.listings.diff()).fillna(0).astype(int)
funnel["share of the file"] = funnel.listings / funnel.listings.iloc[0]
funnel["share removed at this step"] = funnel.removed / funnel.listings.shift().fillna(funnel.listings.iloc[0])
print("4c. From the full file to the scope (every step)")
display(funnel.style.format({"listings": "{:,.0f}", "removed": "{:,.0f}",
                             "share of the file": "{:.2%}", "share removed at this step": "{:.2%}"}))

buckets = final.role_bucket.value_counts(normalize=True)
print("\n4c. The scope by role bucket")
display(buckets.to_frame("share of the scope").style.format("{:.1%}"))

programs = final.tripleten_program.value_counts(normalize=True).reindex(PROGRAM_ORDER).dropna()
print("\n4c. The scope by TripleTen program (adjacent tech role = no program trains for it)")
display(programs.to_frame("share of the scope").style.format("{:.1%}"))

unmapped = final.role_bucket.eq("(unmapped 15- code)")
print(f"\nscope listings whose O*NET code maps to no role bucket: {unmapped.mean():.2%}")
if unmapped.any():
    print("4c. Unmapped O*NET codes in the scope (add them to src/career_switch.py if they're large)")
    display(final.onet_code[unmapped].value_counts(normalize=True).head(10).to_frame("share of unmapped listings"))

**Which occupations are in the scope?** The scope is the TripleTen roles plus adjacent tech roles, found from O*NET codes, so it can include an occupation that is not a tech path a career changer would take. The table below lists every occupation in the scope with its code, its name and its size. The test for each: would a TripleTen graduate, or someone moving from IT support or networking, plausibly apply? An occupation that fails it is dropped in 4c-1, next to credentialed math, because the scope's definition lives here.

In [ ]:
# 4c-4 Every occupation in the scope: its O*NET code and name, how many listings, and the role bucket it falls in
occupations = (final.groupby(["onet_code", "occupation", "role_bucket"], observed=True).size().rename("listings")
               .reset_index().sort_values("listings", ascending=False))
occupations["share of the scope"] = occupations.listings / len(final)
print(f"4c. The {len(occupations)} occupations in the scope")
display(occupations.style.format({"listings": "{:,.0f}", "share of the scope": "{:.2%}"}).hide(axis="index"))

**What the data shows**

- **The scope is 878,937 listings, 6.5% of the file:** computer and mathematical is 7.2% of English US listings, the two added TripleTen roles add 0.69%, and credentialed math (0.05%) is set aside. The O*NET-code fallback adds nothing (the group is always present), and every scope code maps to a role bucket.
- **Three occupations that aren't a tech path are dropped** (29,354 listings, 3.2% of the TripleTen and adjacent roles): search marketing strategists (16,590), GIS technologists (8,758) and document management specialists (4,006). 4c-4 lists the 37 occupations that remain.
- **Software development is the largest bucket (29.8%),** then networks and systems (19.1%), IT project management (12.9%) and data science (11.9%); technical sales is the smallest named bucket (1.1%), and the catch-all "other computer roles" is 0.4%.
- **By TripleTen program:** AI Software Engineering 28.6%, Data Analytics 13.1%, Cybersecurity 5.1%, AI & Machine Learning 4.9%, Quality Assurance 3.8%, UX/UI Design 1.2%, Forward Deployed Engineering / AI Automation 1.1%; **42.2% are adjacent tech roles.**

### 4d Fields only the scope carries

From the stream's full records: the **full description** (`scope_descriptions.parquet`, for EDA_SDA_3), an **anonymous recruiter key** and title (names and contacts aren't kept), **pay as written** (parsed into hourly pay where missing), **location as written** and the **url**. These exist only for the scope's listings (see "Not here yet, on purpose" in section 2).

In [ ]:
# 4d Scope-only fields from the stream's scope parts (every original field); descriptions written to disk
from src.listing_columns import pay_from_text
from src.rows_table import ids_at, scope_fields

start = time.time()
DESC_PATH = LEAN_DIR / "scope_descriptions.parquet"
ids = ids_at(final.index)                 # listing ids for the scope's row positions
extras = scope_fields(ids, DESC_PATH)     # recruiter title, pay as written, location, url, record types; writes the descriptions
final = final.join(extras)
del extras

# Pay from the text where salary_data had none (same conversion and bounds as 2d)
needs = final.salary_hourly_usd.isna() & final.salary_text.notna()
parsed = final.loc[needs, "salary_text"].map(pay_from_text).dropna()
low, high, unit = (parsed.str[k] for k in range(3))
hourly_from_text = (low + high) / 2 * unit.map(PER_YEAR) / HOURS_PER_YEAR
hourly_from_text = hourly_from_text[hourly_from_text.between(10_000 / HOURS_PER_YEAR, 1_000_000 / HOURS_PER_YEAR)]
final["pay_from_text"] = False
final.loc[hourly_from_text.index, "salary_hourly_usd"] = hourly_from_text.astype("float32")
final.loc[hourly_from_text.index, "pay_from_text"] = True

not_found = final.record_type.isna()
if not_found.mean() > 0.01:
    print(f"NOTE: {not_found.mean():.1%} of in-scope listings aren't in the stream's saved full records. If they're the two added TripleTen "
          "roles, run `python -m src.stream_full --extra` (about an hour), then rerun from 4d.")

print("4d. Scope-only fields: share of in-scope listings with each")
display(pd.Series({
    "found in the stream's scope parts": final.record_type.notna().mean(),
    "pay as written": final.salary_text.notna().mean(),
    "location as written": final.location_text.notna().mean(),
    "hourly pay before parsing the text": final.salary_hourly_usd.notna().mean() - final.pay_from_text.mean(),
    "hourly pay added from the text": final.pay_from_text.mean(),
}).round(4).to_frame("share"))
print(f"descriptions saved to {DESC_PATH.name} ({DESC_PATH.stat().st_size / 1e9:.2f} GB) in {time.time() - start:.0f}s")

**What the data shows**

- **Every in-scope listing was found in the stream's full records;** the descriptions are saved for EDA_SDA_3 (2.2 GB of text).
- **Pay as written exists for 48.8%,** and parsing it added hourly pay for 1.5% of listings (46.7% to 48.2%). Every listing carries a location as written.

### 4e The in-scope table at a glance

In [ ]:
# 4e-1 The scope at a glance, and how complete the key fields are
TOP = 15  # how many labels to list in the tables below

print("4e. The in-scope table")
display(pd.Series({
    "listings": f"{len(final):,}",
    "companies": f"{final.company_id.nunique():,}",
    "role buckets": final.role_bucket.nunique(),
    "full months": final.loc[~final.partial_month, "month"].nunique(),
    "late-March listings (partial_month)": f"{final.partial_month.mean():.1%}",
}).to_frame("value"))

KEY_FIELDS = {
    "first and last seen": final.first_seen.notna() & final.last_seen.notna(),
    "role bucket": final.role_bucket.ne("(unmapped 15- code)"),
    "source": final.source_type.notna() & final.source_type.ne("(missing)"),
    "description": final.has_desc.fillna(False).astype(bool),
    "seniority": final.seniority.notna() & final.seniority.ne("not_set"),
    "hourly pay": final.salary_hourly_usd.notna(),
    "employer's posting date": final.posted.notna(),
    "years of experience stated": final.years_required.notna(),
}
print("\n4e. Share of in-scope listings with each key field")
display(pd.DataFrame(KEY_FIELDS).mean().round(3).to_frame("share filled"))

In [ ]:
# 4e-2 Contract types (messy: a listing can have several, and the labels are free text)
# Each label is its own True/False column from 2a-3, so a label's share of the scope is just that column's mean.
contract_columns = [c for c in final.columns if c.startswith("contract_")]
contract = final[contract_columns].mean().sort_values(ascending=False).head(TOP).round(3).to_frame("share")
contract.loc["(none)"] = round((~final[contract_columns].any(axis=1)).mean(), 3)  # listings with no contract label at all
print("4e. contract_types (a listing can have several)")
display(contract)
print(f"in-scope listings whose company is in the companies table: {final[COMPANY].isin(companies['id']).mean():.1%}")

**What the data shows**

- **The in-scope table is 878,937 listings from 68,863 companies.** Timing, role bucket, source and seniority are complete; description 82%, pay 48%, years stated 46%, employer posting date 42%.
- **Contract types are messy** (free-text labels; full time 55.0%, remote 21.9%, hybrid 20.6%, onsite 19.8%, contract 8.7%, none 17.6%). Every scope company is in the companies table.
- **A correction to `EDA_SDA_1`:** its text said "none 27%" and left out remote, for the same reason as in 4b (the packed column kept only 63 of 161 labels). Its description, pay, years-stated and posting-date shares (78%, 45%, 43%, 39%) were also out of step with its own printed tables (82%, 48%, 46%, 42%); the figures above match this notebook's run.

## 5. Q2: Which factors should drive prioritization?

**In-scope listings only** (originals, `so`; repost rates use every listing). A factor only helps ranking if it **varies**: 5a shows the main factors by role bucket and program, with each one's spread (max / min); 5b–5h look closer at one family each.

In [ ]:
# 5.0 The scope's original listings, the role buckets in display order, and the two small table-display helpers used below
from src.career_switch import ROLE_ORDER

so = final[~final.is_repost]  # originals only: a repost would count the same job twice
BUCKETS = [b for b in ROLE_ORDER + ["other computer roles"] + EXTRA_ORDER + ["(unmapped 15- code)"] if (so.role_bucket == b).any()]
PROGRAMS = [p for p in PROGRAM_ORDER if (so.tripleten_program == p).any()]
print(f"scope: {len(final):,} listings ({len(so):,} originals) in {len(BUCKETS)} role buckets")


def shade(table, formats, highlight=None):
    """Display a table: one format per column (formats is a dict, or one string for every column), each numeric
    column shaded on its own scale (a share and a pay can't share one), and the row named `highlight` outlined."""
    styled = table.style.format(formats)
    if highlight in table.index:
        outline = f"font-weight: bold; border-top: 2px solid {ORANGE}; border-bottom: 2px solid {ORANGE}"
        styled = styled.apply(lambda row: [outline if row.name == highlight else "" for _ in row], axis=1)
    for c in table.columns:
        if pd.api.types.is_numeric_dtype(table[c]) and table[c].notna().sum() > 1 and table[c].nunique() > 1:
            styled = styled.background_gradient(cmap="Blues", subset=[c], low=0.05, high=0.35)
    display(styled)


def format_rows(table, formats):
    """Return a copy of the table as text, where each ROW has its own format (formats: row name -> format string).
    For tables whose rows are different measures (a pay, a count, a share) so no one column format fits."""
    shown = table.astype(object)
    for row, fmt in formats.items():
        for col in table.columns:
            value = table.loc[row, col]
            shown.loc[row, col] = fmt.format(value) if pd.notna(value) else ""
    return shown

### 5a The main factors, by role bucket and program

In [ ]:
# 5a-1 The main factors for one group of originals (g)
def targets(g):
    """One row of numbers per group: the factors a ranking could use."""
    curve = km.km_curve(g.days_open, ~g.live)
    return pd.Series({
        "median days open (Kaplan-Meier)": curve[curve <= 0.5].index.min(),
        "new postings per week": g.full_week.sum() / FULL_WEEKS,
        "salary disclosed": g.salary_hourly_usd.notna().mean(),
        "median hourly pay (USD)": g.salary_hourly_usd.median(),
        "mentions remote": g.mentions_remote.mean(),
        "median words per description": g.desc_words.where(g.has_desc.fillna(False).astype(bool)).median(),
        "manager-level seniority": (g.seniority == "manager").mean(),
        "entry-level seniority": g.entry_seniority.mean(),
        "open to career changers (any sign)": g.career_open.mean(),
    })


TARGET_FORMATS = {"median days open (Kaplan-Meier)": "{:.1f}", "new postings per week": "{:,.0f}", "salary disclosed": "{:.1%}",
                  "median hourly pay (USD)": "${:,.2f}", "mentions remote": "{:.1%}", "median words per description": "{:,.0f}",
                  "manager-level seniority": "{:.1%}", "entry-level seniority": "{:.1%}",
                  "open to career changers (any sign)": "{:.1%}"}

by_bucket = pd.DataFrame({b: targets(so[so.role_bucket == b]) for b in BUCKETS}).T
# The range across buckets leaves out "other computer roles": since the three non-tech occupations were dropped (4c-1) it is a
# catch-all of 3.8K listings (0.4% of the scope), too small and mixed to say what a typical bucket looks like
named = by_bucket.drop(index="other computer roles", errors="ignore")
overview = pd.DataFrame({
    "whole scope": targets(so),
    "bucket min": named.min(),
    "bucket median": named.median(),
    "bucket max": named.max(),
})
# How much a factor differs between buckets: the highest bucket divided by the lowest (a 0 lowest has no ratio)
overview["spread (max / min)"] = overview["bucket max"] / overview["bucket min"].replace(0, np.nan)

# Each row of `overview` is a different measure, so each row gets its own format; the spread column is always "x.x"
overview_shown = format_rows(overview, TARGET_FORMATS)
overview_shown["spread (max / min)"] = overview["spread (max / min)"].map("{:.1f}".format)
print("5a. The main factors: the whole scope, and the range across role buckets")
display(overview_shown)

In [ ]:
# 5a-2 The same factors by role bucket, then by TripleTen program
print("5a. The main factors by role bucket (each column shaded on its own scale)")
shade(by_bucket, TARGET_FORMATS)

by_program = pd.DataFrame({p: targets(so[so.tripleten_program == p]) for p in PROGRAMS}).T
print("\n5a. The same factors by TripleTen program (adjacent tech role = no program trains for it)")
shade(by_program, TARGET_FORMATS, highlight="adjacent tech role")

In [ ]:
# 5a-3 Chart: how much each factor differs between role buckets
spread = overview["spread (max / min)"].dropna().sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh([str(name)[:42] for name in spread.index], spread.values, color=BLUE, height=0.65)
for row, value in enumerate(spread.values):
    ax.text(value, row, f" {value:.1f}", va="center", color=INK_2, fontsize=8.5)  # the value at the end of each bar
ax.axvline(1, color=INK_2, lw=1)  # a spread of 1 would mean no difference between buckets
ax.set(title="How much each factor differs between role buckets", xlabel="spread: highest bucket / lowest bucket")
ax.grid(axis="y", visible=False)
ax.margins(x=0.18)
plt.tight_layout()
plt.show()

**What the data shows**

- **Volume and manager-level seniority vary most between buckets** (27 and 25 times; the range leaves out the 3.8K-listing catch-all "other computer roles"): technical sales gets 340 new listings a week and software development 9,200; manager-level is 1.7% in IT support and 43.7% in IT project management. Of the accessibility measures, entry-level seniority varies 10.8 times and open to career changers 4.8.
- **Pay varies 2.4 times** ($31.10 an hour in IT support to $73.54 in software development), **while timing varies only 1.7** (medians of 22 to 38 days).
- **By program:** Data Analytics roles are the most open (28.9% show any sign) at $49.28 an hour; AI Software Engineering and AI & Machine Learning pay the most (about $74) but are among the least open (18 to 19%).

### 5b Application window

Share still open N days after first seen (Kaplan–Meier, so still-open listings count as unfinished): the basis for an **apply-by estimate**.

In [ ]:
# 5b-1 Application window by role bucket: the share still open N days after first seen
DAYS = [3, 7, 14, 30, 60]


def open_after(g, days=DAYS):
    """The Kaplan-Meier share of a group's listings still open after each of the given days."""
    curve = km.km_curve(g.days_open, ~g.live)
    return [km.survival_at(curve, d) for d in days]


window = {b: open_after(so[so.role_bucket == b]) for b in BUCKETS}
window["(whole scope)"] = open_after(so)
window_table = pd.DataFrame(window, index=[f"open after {d}d" for d in DAYS]).T
print("5b. Share of listings still open N days after first seen (Kaplan-Meier)")
shade(window_table, "{:.1%}", highlight="(whole scope)")

In [ ]:
# 5b-2 Chart: survival curves for the fastest and slowest role buckets (every other bucket in grey)
grid = np.arange(0, 121)  # days 0 to 120


def on_grid(curve):
    """Put a survival curve on the same 0-120 day grid so curves from different groups line up (flat between events)."""
    return curve.reindex(curve.index.union(grid)).ffill().reindex(grid).fillna(1.0)


medians = {b: km.median_survival(so.days_open[so.role_bucket == b], ~so.live[so.role_bucket == b]) for b in BUCKETS}
medians = {b: m for b, m in medians.items() if pd.notna(m)}
fast, slow = min(medians, key=medians.get), max(medians, key=medians.get)

fig, ax = plt.subplots(figsize=(9.5, 4.2))
for b in BUCKETS:
    if b not in (fast, slow):
        g = so[so.role_bucket == b]
        ax.step(grid, on_grid(km.km_curve(g.days_open, ~g.live)), where="post", color=MUTED, lw=1)
ax.plot([], [], color=MUTED, lw=1, label="other role buckets")  # an empty line, only so the legend has an entry for the grey ones
for b, color, tag in ((fast, ORANGE, "fastest"), (slow, BLUE, "slowest")):
    g = so[so.role_bucket == b]
    ax.step(grid, on_grid(km.km_curve(g.days_open, ~g.live)), where="post", color=color, label=f"{b} ({tag})")
ax.step(grid, on_grid(km.km_curve(so.days_open, ~so.live)), where="post", color=INK, lw=1.5, ls="--", label="whole scope")
ax.set(title="Share of listings still open, by days since first seen", xlabel="days since first seen",
       ylabel="share still open", xlim=(0, 120), ylim=(0, 1.02))
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.show()

In [ ]:
# 5b-3 Check: timing with and without PL crawl artifacts (scope originals; flags from 2e)
views = {"every listing": pd.Series(True, index=so.index),
         "without batch first sightings": ~so.batch_first_seen,
         "without PL crawl dropouts": ~so.crawl_dropout,
         "without either": ~so.batch_first_seen & ~so.crawl_dropout}
artifact_check = pd.DataFrame({name: {
    "share of listings": keep.mean(),
    "median days open (Kaplan-Meier)": km.median_survival(so.days_open[keep], ~so.live[keep]),
    "open after 7d": open_after(so[keep], [7])[0],
    "open after 30d": open_after(so[keep], [30])[0],
} for name, keep in views.items()}).T
print("5b check: timing with and without PL crawl artifacts")
display(artifact_check.style.format({"share of listings": "{:.1%}", "median days open (Kaplan-Meier)": "{:.1f}",
                                     "open after 7d": "{:.1%}", "open after 30d": "{:.1%}"}))

**What the data shows**

- **The window is short:** 28.7% of in-scope listings close within a week, the median is 24.1 days, and 44.8% are still open after 30 days.
- **Most buckets move alike** (67 to 73% still open after a week). **Technical sales is the slow outlier** (84% after a week, median 38 days); IT support, database, quality assurance and other computer roles are the fastest (median 22 days).
- **PredictLeads crawl artifacts barely move timing:** without them the median falls from 24.1 to 23.0 days, and the share open after a week stays at 71%.

### 5c Competition: company hiring volume

Volume is the company's listings **in any field first seen before each listing** (`company_earlier`), as the model will see it.

In [ ]:
# 5c-1 Company concentration by role bucket (in-scope listings)
def concentration(g):
    """How concentrated a group's listings are in a few companies."""
    per_co = g[COMPANY].value_counts()
    per_co = per_co[per_co > 0]  # company_id is a category: drop companies with no listings in this group
    return pd.Series({
        "companies": per_co.size,
        "postings per company (median)": per_co.median(),
        "top 1% of companies' share": per_co.head(max(1, per_co.size // 100)).sum() / len(g),
        "top 10% of companies' share": per_co.head(max(1, per_co.size // 10)).sum() / len(g),
        "companies with 5+ postings": (per_co >= 5).sum(),
    })


concentration_table = (so[[COMPANY, "role_bucket"]].groupby("role_bucket", observed=True)
                       .apply(concentration, include_groups=False).reindex(BUCKETS))
print("5c. Company concentration by role bucket")
shade(concentration_table, {"companies": "{:,.0f}", "postings per company (median)": "{:.1f}",
                            "top 1% of companies' share": "{:.1%}", "top 10% of companies' share": "{:.1%}",
                            "companies with 5+ postings": "{:,.0f}"})

In [ ]:
# 5c-2 Do high-volume companies' listings close faster or slower? (earlier listings in any field: company_earlier, 2e)
BANDS = ["1-4", "5-49", "50-499", "500+"]


def size_band(listings):
    """A company's listing count as one of the four volume bands (a missing count is read as 1)."""
    return pd.cut(listings.fillna(1), [0, 4, 49, 499, np.inf], labels=BANDS)


# company_earlier counts the listings seen BEFORE this one, so +1 is the company's volume including this listing
asof_band = size_band(so.company_earlier + 1)
since_may = so.first_seen >= pd.Timestamp("2026-05-01", tz="UTC")  # a month or more of history behind each listing


def median_by_band(keep):
    """Kaplan-Meier median days open within each volume band, for the listings in `keep`."""
    return so.loc[keep, ["days_open", "live"]].groupby(asof_band[keep], observed=True).apply(
        lambda g: km.median_survival(g.days_open, ~g.live))


band_table = pd.DataFrame({
    "share of listings": asof_band.value_counts(normalize=True),
    "median days open (Kaplan-Meier)": median_by_band(pd.Series(True, index=so.index)),
    "median days, first seen May or later": median_by_band(since_may),
    "repost rate": final.is_repost.groupby(size_band(final.company_earlier + 1), observed=True).mean(),
}).reindex(BANDS)
print("5c. Median days open (Kaplan-Meier) by company volume (earlier listings in any field)")
display(band_table.style.format({"share of listings": "{:.1%}", "median days open (Kaplan-Meier)": "{:.1f}",
                                 "median days, first seen May or later": "{:.1f}", "repost rate": "{:.1%}"}))

### Why we need the Gehan-Wilcoxon test

The table above shows medians for four groups of companies, from 45 days at the lowest-volume companies down to 12 at the highest. A median alone can't say whether gaps that size could just be chance, and a plain test on days open would throw away the still-open listings (the same mistake section 3 explains). So the next two cells use a **Gehan-Wilcoxon test**, the survival version of a Wilcoxon test: it compares Kaplan-Meier curves, giving the most weight to early days, when most listings are still being watched and most of them close.

It is run two ways. **All four bands at once** asks whether any of the four curves differ from the others, which is what the chart shows. **Every pair of bands** (six pairs) asks whether each step is real, and not just the two ends. First the three neighboring steps: 1-4 vs. 5-49, 5-49 vs. 50-499 and 50-499 vs. 500+. Then the pairs two steps apart: 1-4 vs. 50-499 and 5-49 vs. 500+. Last, the two ends: 1-4 vs. 500+.

How to read the results: a **p-value** is the chance of seeing a gap this large if the groups really closed at the same rate. A tiny p-value means the gap is real, **not that it is big**; the size is in the medians (45, 34, 28 and 12 days). It also doesn't say company volume *causes* faster closing, since big companies differ in other ways; `EDA_SDA_4`'s Cox model holds those fixed. The method and a worked example are in `docs/gehan_wilcoxon.md`.

In [ ]:
# 5c-3 Chart: survival curves by company volume (earlier listings only), with a Gehan-Wilcoxon test across all four bands
# Each curve is the Kaplan-Meier share of listings still open after each day; a median is where a curve crosses 50%.
# Days open is censored (still-open listings haven't finished), so the test is the survival version of a Wilcoxon test:
# a log-rank test weighted toward early differences (Gehan-Breslow-Wilcoxon). Formal tests are in EDA_SDA_4; this is a preview.
# The tests run on up to 500,000 random listings per band; the curves use every listing.
from lifelines.statistics import multivariate_logrank_test, pairwise_logrank_test

TEST_N = 500_000
fig, ax = plt.subplots(figsize=(9, 4.2))
samples = []
for band, color in zip(BANDS, [MUTED, AQUA, ORANGE, BLUE]):
    g = so.loc[asof_band == band, ["days_open", "live"]]
    if g.empty:
        continue
    curve = km.km_curve(g.days_open, ~g.live)
    ax.step(curve.index, 100 * curve.values, where="post", color=color, lw=1.8,
            label=f"{band} (median {km.median_survival(g.days_open, ~g.live):.0f} days, {len(g):,} listings)")
    samples.append(g.sample(min(len(g), TEST_N), random_state=0).assign(band=band))  # this band's test sample
ax.axhline(50, color=INK_2, ls="--", lw=0.8)
ax.text(1, 51.5, "median: half still open", color=INK_2, fontsize=8)

sample = pd.concat(samples)
title = "Share still open by company volume (earlier listings in any field)"
all_bands = multivariate_logrank_test(sample.days_open, sample.band, ~sample.live, weightings="wilcoxon")
p_text = "< 0.001" if all_bands.p_value < 0.001 else f"= {all_bands.p_value:.3f}"
title += f"\nGehan-Wilcoxon test, all {len(samples)} bands: chi-square {all_bands.test_statistic:,.0f} ({len(samples) - 1} degrees of freedom), p {p_text}"
ax.set(title=title, xlabel="days since first seen", ylabel="% still open", xlim=(0, 120), ylim=(0, 100))
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# 5c-4 Is each step between bands real? The same Gehan-Wilcoxon test on every pair of bands (same samples as 5c-3)
# Six pairs means six tests, so by chance alone one could look "significant"; the p-values here are far below that worry.
pairs = pairwise_logrank_test(sample.days_open, sample.band, ~sample.live, weightings="wilcoxon").summary
pair_table = pd.DataFrame([
    {"band A": a, "band B": b, "bands apart": abs(BANDS.index(b) - BANDS.index(a)), "chi-square": row.test_statistic,
     "p-value": "< 0.001" if row.p < 0.001 else f"{row.p:.3f}"}
    for (a, b), row in pairs.iterrows()])
# Neighboring bands first (1 apart), then 2 apart, then the two ends (3 apart); within each, lowest band first
pair_table = pair_table.sort_values("bands apart", kind="stable")
print("5c. Gehan-Wilcoxon test on every pair of volume bands")
display(pair_table.style.format({"chi-square": "{:,.0f}"}).hide(axis="index"))

**How to read the chi-square column.** Each chi-square measures how far two bands' closing patterns are from what we'd expect if they closed at the same rate: **0 means identical curves, and bigger means a bigger gap.** It is a score of evidence, not a size: it has no units, and it grows with the number of listings, so with hundreds of thousands per band it comes out in the thousands.

To turn a chi-square into a verdict, compare it with a cutoff. For a single pair (1 degree of freedom), above **3.84** means the gap is unlikely to be chance at the usual 5% level (p < 0.05), and above **10.83** means p < 0.001. For the four-band test in the chart title (3 degrees of freedom) the cutoffs are 7.81 and 16.27. Our values are far above all of these, so every p-value reads "< 0.001": the steps between bands are real, not luck. Use the chi-square to compare rows (a pair at 20,000 differs more strongly than one at 2,000), and use the medians (45, 34, 28, 12 days) for how big the differences are.

**What the data shows**

- **Company volume is the strongest timing signal:** counting only earlier listings in any field, in-scope listings at companies with 500+ listings close in a median **12 days, vs. 45 at companies with 1 to 4**. The four bands differ (Gehan-Wilcoxon test across all four; the chi-square and p are in the chart title), and the pairwise table shows each step between bands separately.
- **The gradient is steady across bands** (45, 34, 28 and 12 days) and holds for listings first seen from May (56 vs. 12 days).
- **High-volume employers dominate:** the top 10% of companies post 47 to 74% of each bucket's listings, and reposting rises with volume (2.4% at 1 to 4 listings to 8.7% at 500+).

### 5d Source

Hiring platform, job board or company site: share, arrivals and window. Job boards are under-represented in the PredictLeads crawl.

In [ ]:
# 5d-1 Sources: share of postings, arrival rate, window
print("5d. Source type")
display(so.source_type.value_counts(normalize=True).to_frame("share").style.format("{:.1%}"))

# The platform where there is one (Workday, Greenhouse, ...), otherwise the source type
source = so.platform.astype(object).fillna(so.source_type.astype(object)).astype("category")
by_source = so[["days_open", "live", "full_week"]].groupby(source, observed=True).apply(lambda g: pd.Series({
    "postings": len(g),
    "share": len(g) / len(so),
    "new per week": g.full_week.sum() / FULL_WEEKS,
    "median days open (Kaplan-Meier)": km.median_survival(g.days_open, ~g.live),
    "open after 7d": open_after(g, [7])[0],
}))
print("\n5d. Platforms and other sources: volume, arrival rate, window (the 15 largest)")
display(by_source.sort_values("postings", ascending=False).head(15).style.format({
    "postings": "{:,.0f}", "share": "{:.1%}", "new per week": "{:,.0f}",
    "median days open (Kaplan-Meier)": "{:.1f}", "open after 7d": "{:.1%}"}))

In [ ]:
# 5d-2 Source mix within each role bucket: of a bucket's listings, what share come from each source
mix = pd.crosstab(so.role_bucket, source, normalize="index").reindex(BUCKETS)
top_sources = mix.sum().sort_values(ascending=False).index[:8]  # the eight sources with the most listings overall
print("5d. Source mix within each role bucket (share of the bucket's listings)")
shade(mix[top_sources], "{:.1%}")

**What the data shows**

- **62.7% of in-scope listings are on company sites or other domains,** 29.6% on hiring platforms and 7.8% on job boards.
- **Windows differ widely by source:** LinkedIn, Oracle and Taleo listings close in a median 14.5 to 15.7 days, company sites 21, Workday 32, Greenhouse 49, Lever 67 and Ashby 72.
- **The mix is similar in every bucket** (company sites 53 to 66%), so source separates listings, not roles. Platform choice tracks employer size, so source is tested alongside company volume.
- **A correction to `EDA_SDA_1`:** its written shares (65%, 28%, 7.1%, and 56 to 70% across buckets) don't match its own printed tables (62%, 30%, 7.9%); the figures above match both this notebook's run and the original's tables.

### 5e Competition: reposts

A repost is the same job listed again under a new id: how common, by bucket and source, and how soon after the original.

In [ ]:
# 5e Repost rates by role bucket and by source, and days from the original to each repost
def repost_rate(key):
    """The number and share of reposts among every in-scope listing (reposts included), grouped by `key`."""
    return final.groupby(key, observed=True).is_repost.agg(reposts="sum", repost_rate="mean")


print("5e. Repost rate by role bucket (every in-scope listing)")
display(repost_rate("role_bucket").reindex(BUCKETS).style.format({"reposts": "{:,.0f}", "repost_rate": "{:.1%}"}))
print("\n5e. Repost rate by source type")
display(repost_rate("source_type").style.format({"reposts": "{:,.0f}", "repost_rate": "{:.1%}"}))

# The original is the earliest listing with the same fingerprint (same company, title and description)
first = final[~final.is_repost & final.repost_fp.notna()].groupby("repost_fp").first_seen.min()
reposts = final.loc[final.is_repost, ["repost_fp", "first_seen"]].join(first.rename("original_seen"), on="repost_fp")
days_to_repost = (reposts.first_seen - reposts.original_seen).dt.total_seconds() / 86400
print("\n5e. Days from the original to each repost")
display(days_to_repost.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_frame("days").style.format("{:,.1f}"))

**What the data shows**

- **6.5% of in-scope listings are reposts:** most in technical sales (9.4%) and business and data analysis (9.4%), fewest in quality assurance (4.6%) and IT support (4.9%).
- **Job boards repost far more than other sources** (24.3% of their listings, vs. 4.1% on hiring platforms and 4.9% on company sites).
- **Reposts come back fast:** a median 7.6 days after the original, and a quarter within a day, so many are re-listings or PredictLeads crawler refreshes rather than a failed hire.

### 5f Recruiters

Anonymous keys only; counts and shares.

In [ ]:
# 5f Recruiters (anonymous keys; aggregates only)
has_recruiter = so.recruiter_key.notna()
per_recruiter = so.recruiter_key.dropna().value_counts()                   # listings per recruiter key
recruiter_size = so.recruiter_key.map(per_recruiter)                        # each listing's recruiter's total listings
recruiter_band = pd.cut(recruiter_size, [0, 1, 4, np.inf], labels=["1 listing", "2-4", "5+"])

print("5f. Recruiters in the scope")
display(pd.Series({
    "share of listings naming a recruiter": f"{has_recruiter.mean():.1%}",
    "recruiters (anonymous keys)": f"{per_recruiter.size:,}",
    "recruiters with 5+ listings": f"{(per_recruiter >= 5).sum():,}",
    "share of recruiter listings from recruiters with 5+": f"{per_recruiter[per_recruiter >= 5].sum() / max(per_recruiter.sum(), 1):.1%}",
    "share with a recruiter title": f"{so.recruiter_title.notna().mean():.1%}",
}).to_frame("value"))

groups = {"named recruiter": has_recruiter, "no recruiter named": ~has_recruiter}
groups.update({f"recruiter with {band}": recruiter_band.eq(band) for band in ["1 listing", "2-4", "5+"]})
recruiter_window = pd.DataFrame({name: {
    "listings": int(keep.sum()),
    "median days open": km.median_survival(so.days_open[keep], ~so.live[keep]) if keep.any() else np.nan,
    "open after 7d": open_after(so[keep], [7])[0] if keep.any() else np.nan,
} for name, keep in groups.items()}).T
print("\n5f. Window by recruiter (Kaplan-Meier)")
display(recruiter_window.style.format({"listings": "{:,.0f}", "median days open": "{:.1f}", "open after 7d": "{:.1%}"}))

**What the data shows**

- **Only 1.3% of in-scope listings name a recruiter:** 5,139 anonymous recruiters; the 432 with 5+ listings post 38% of recruiter listings.
- **Recruiter listings close much faster:** a median 11.2 days vs. 24.4 without one; for recruiters with 5+ listings, **6.5 days** (48% still open after a week vs. 71%).
- **Many recruiter listings may come from staffing agencies,** so EDA_SDA_2 tests recruiters against its staffing-agency flag before they're used.

### 5g Who is open to newcomers

The text rules behind the career-change flags, among listings with a description.

In [ ]:
# 5g Career-change accessibility: the text rules, by role bucket
def access(g):
    """Text measures for one group, among listings with a description."""
    described = g[g.has_desc]
    stated = described.years_required.dropna()
    return pd.Series({
        "states years required": described.years_required.notna().mean(),
        "median years required (where stated)": stated.median(),
        "asks 2 years or fewer (where stated)": (stated <= 2).mean(),
        "entry-level or training wording": described.entry_wording.mean(),
        "career-changer wording": described.changer_wording.mean(),
        "degree required": described.degree_required.mean(),
        "or equivalent experience": described.equivalent_experience.mean(),
        "median hourly pay (USD), open listings only": g.loc[g.career_open.fillna(False), "salary_hourly_usd"].median(),
    })


ACCESS_COLS = ["has_desc", "years_required", "entry_wording", "changer_wording", "degree_required",
               "equivalent_experience", "career_open", "salary_hourly_usd"]
access_table = pd.DataFrame({b: access(so.loc[so.role_bucket == b, ACCESS_COLS]) for b in BUCKETS}).T
access_table.loc["whole scope"] = access(so[ACCESS_COLS])
print("5g. Career-change text rules by role bucket (each column shaded on its own scale)")
shade(access_table, {"states years required": "{:.1%}", "median years required (where stated)": "{:.1f}",
                     "asks 2 years or fewer (where stated)": "{:.1%}", "entry-level or training wording": "{:.1%}",
                     "career-changer wording": "{:.1%}", "degree required": "{:.1%}", "or equivalent experience": "{:.1%}",
                     "median hourly pay (USD), open listings only": "${:,.2f}"}, highlight="whole scope")

**What the data shows**

- **IT support is the entry door:** 48% of its listings that state years ask for 2 or fewer, and 22% are entry level (5a). Then business and data analysis (33% show any sign). The small catch-all "other computer roles" (3.8K listings) shows the most openness (65%) but is too small and mixed to read as a door. IT project management is the hardest (2.1% entry level, 9.1% open).
- **Open listings pay less, but still well:** a median $38.46 an hour, vs. $64.90 across the scope ($26.00 for open IT support listings). Degree requirements are rare (3.9%), but 61.3% of listings that state years ask for 5 or more.

### 5h Do the flags go with how long listings stay open?

Each row compares listings with and without a flag, among those where it's known. Keyword flags misfire, so EDA_SDA_3 checks them; urgency is left to its NLP.

In [ ]:
# 5h-1 Do the flags relate to how long postings stay open?
flags = {
    "salary disclosed": so.salary_hourly_usd.notna(),
    "mentions remote": so.mentions_remote,
    "long description (top 25%)": so.desc_words > so.desc_words.quantile(0.75),
    "entry-level seniority": so.entry_seniority,
    "asks 2 years or fewer": (so.years_required <= 2).where(so.years_required.notna()).astype("boolean"),  # only listings that state years
    "asks 5 years or more": (so.years_required >= 5).where(so.years_required.notna()).astype("boolean"),
    "entry-level or training wording": so.entry_wording,
    "degree required": so.degree_required,
    "open to career changers (any sign)": so.career_open,
}
flag_rows = {}
for name, flag in flags.items():
    known = flag.notna().to_numpy()                  # compare with vs. known-without, never vs. unknown
    has = flag.fillna(False).astype(bool).to_numpy()
    with_flag = open_after(so[has], [7, 30])
    without_flag = open_after(so[known & ~has], [7, 30])
    flag_rows[name] = {"share of listings": has[known].mean(), "open 7d (with)": with_flag[0], "open 7d (without)": without_flag[0],
                       "open 30d (with)": with_flag[1], "open 30d (without)": without_flag[1]}
flag_table = pd.DataFrame(flag_rows).T
print("5h. Flags vs. application window (scope)")
display(flag_table.style.format("{:.1%}"))

In [ ]:
# 5h-2 Chart: share still open after 7 days, with vs. without each flag
feat = flag_table.sort_values("open 7d (with)")
fig, ax = plt.subplots(figsize=(8.5, 4.4))
y = np.arange(len(feat))
ax.hlines(y, feat["open 7d (without)"], feat["open 7d (with)"], color=MUTED, lw=2)
ax.scatter(feat["open 7d (without)"], y, color=INK_2, s=60, zorder=3, label="without the feature")
ax.scatter(feat["open 7d (with)"], y, color=BLUE, s=60, zorder=3, label="with the feature")
ax.set_yticks(y, feat.index)
ax.set(title="Share of listings still open after 7 days", xlabel="share still open")
ax.grid(axis="y", visible=False)
ax.legend(fontsize=8.5, loc="lower right")
plt.tight_layout()
plt.show()

**What the data shows**

- **Disclosed pay and long descriptions go with listings staying open longer** (79% and 81% still open after a week, vs. 64% and 68% without). **Remote wording shows no gap** (78% vs. 79%).
- **Listings open to career changers close a bit faster** (75% still open after a week vs. 79%).
- **One feature at a time:** these overlap with company volume and source, so EDA_SDA_4's Cox model separates them.

## 6. Q3: What time information can prioritization rely on?

Six months of data rules out seasons. Checks: history and weekly arrivals (full weeks only), whether `first_seen` follows the employer or the PredictLeads crawler, which listings count as still open, and whether the date picker is fair.

In [ ]:
# 6a-1 How much history does the data hold, and how many listings arrive per full week?
weekly = so.first_seen.dt.tz_localize(None).dt.to_period("W").value_counts().sort_index()  # new listings in each week
full = weekly[(weekly.index > FIRST_WEEK) & (weekly.index < LAST_WEEK)]                    # drop the partial first and last weeks

print("6a. History and weekly arrivals (full weeks only)")
display(pd.Series({
    "first_seen from": so.first_seen.min().date(),
    "first_seen to": so.first_seen.max().date(),
    "distinct months": so.month.nunique(),
    "complete years of monthly data (seasonal decomposition needs 2+)": so.month.nunique() // 12,
    "full weeks": full.size,
    "median new listings per week": f"{full.median():,.0f}",
    "lowest and highest week, vs. the median": f"{full.min() / full.median():.2f}x and {full.max() / full.median():.2f}x",
}).to_frame("value"))

In [ ]:
# 6a-2 Chart: new listings per week (partial first and last weeks greyed, labelled)
fig, ax = plt.subplots(figsize=(9.5, 3.4))
ax.plot([week.start_time for week in full.index], full.values, color=BLUE, marker="o", ms=4, label="full week")
partial = weekly.drop(full.index)
ax.scatter([week.start_time for week in partial.index], partial.values, color=MUTED, s=40, zorder=3,
           label="partial week (left out of weekly rates)")
ax.set(title="New in-scope listings per week (first seen)", ylabel="listings")
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.show()

In [ ]:
# 6b Employer or PL crawler? posted (the employer's date) vs. first_seen (the date PredictLeads' crawler first saw it)
has_posted = so.posted.notna()
lag = (so.first_seen - so.posted)[has_posted].dt.total_seconds() / 86400  # days from the employer's date to the PL crawler's
print(f"6b. Days from posted to first_seen (listings with posted: {has_posted.mean():.1%})")
display(lag.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_frame("days"))

# If first_seen followed the employer, its weekdays would match the posted weekdays. If they differ, it follows the crawler.
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday = pd.DataFrame({
    "posted (UTC)": so.posted[has_posted].dt.day_name().value_counts(normalize=True),
    # posted timestamps sit at 22:00/23:00 UTC = midnight Central European time, so read them as CET dates
    "posted (CET date)": so.posted[has_posted].dt.tz_convert("Europe/Berlin").dt.day_name().value_counts(normalize=True),
    "first_seen, same listings": so.first_seen[has_posted].dt.day_name().value_counts(normalize=True),
    "first_seen, all listings": so.first_seen.dt.day_name().value_counts(normalize=True),
}).reindex(WEEKDAYS)
print("\n6b. % of listings by weekday (if the columns disagree, first_seen reflects the crawler)")
display((100 * weekday).round(1))

In [ ]:
# 6c Which listings count as still open? Our rule vs. the hackathon app's (merge decision 2)
# Ours: last seen within 2 days of September 30. The app's signals: last seen within 7 days and not marked closed.
# The app's listing filter today: status not "closed" (status is only ever "closed" or empty).
days_since = (DATA_END - so.last_seen).dt.days
not_closed = so.status.ne("closed").fillna(True).astype(bool)
rules = {
    "ours: last seen within 2 days": so.live.astype(bool),
    "last seen within 7 days": days_since <= 7,
    "the app's signals: within 7 days and not closed": (days_since <= 7) & not_closed,
    "the app's listings today: status not closed": not_closed,
}
still_open = pd.DataFrame({
    "share counted open": {name: counted.mean() for name, counted in rules.items()},
    "share also open under ours (2 days)":
        {name: (counted & rules["ours: last seen within 2 days"]).sum() / max(counted.sum(), 1) for name, counted in rules.items()},
    "share last seen 30+ days before Sep 30":
        {name: (counted & (days_since > 30)).sum() / max(counted.sum(), 1) for name, counted in rules.items()},
})
print("6c. Share of in-scope listings counted as still open, by rule")
display(still_open.style.format("{:.1%}"))

In [ ]:
# 6d Date picker check: in-scope listings live on the 1st of each month (shares and medians)
# Live on a date = first seen on or before it and last seen on or after it. The data starts on 2026-03-30, so listings
# that opened earlier are missing (left truncation): early dates should look thin and young.
picker_rows = {}
for date in pd.date_range("2026-04-01", "2026-09-01", freq="MS", tz="UTC"):
    age = (date - so.first_seen[(so.first_seen <= date) & (so.last_seen >= date)]).dt.days  # days since each live listing was first seen
    picker_rows[date.strftime("%Y-%m-%d")] = {"live listings": len(age), "median days since first seen": age.median(),
                                              "share first seen in the past 30 days": (age <= 30).mean()}
picker = pd.DataFrame(picker_rows).T
picker.insert(1, "live, vs. September 1 (share)", picker["live listings"] / picker["live listings"].iloc[-1])
print("6d. Listings live on the 1st of each month")
display(picker.style.format({"live listings": "{:,.0f}", "live, vs. September 1 (share)": "{:.1%}",
                             "median days since first seen": "{:.1f}", "share first seen in the past 30 days": "{:.1%}"}))

**What the data shows**

- **Six months of steady history:** no seasonal advice (7 distinct months, no complete year), but 74.5% of in-scope listings closed by September 30 (25.5% were still open), so timing is well observed. A median **30,303 new in-scope listings arrive per full week**; the quietest week is 0.76x the median and the busiest 1.27x.
- **`first_seen` is a good freshness clock:** it lags the employer's posting date by a median 1.4 days (75% within 3.4). Its weekdays follow the PredictLeads crawler (Friday peak, 19.8%), while posting dates (41% of listings) peak Tuesday to Thursday.
- **Still-open rules (merge decision 2):** our rule (last seen within 2 days) counts 25.5% of in-scope listings as open, and the app's signal rule (7 days, not closed) 25.4%, mostly the same listings (94% overlap). **The app's current listing filter (status not closed) counts 39.6%, and 26.2% of those were last seen 30+ days before September 30.**
- **The date picker is fair from July:** live listings reach 80% of September's level, and the share first seen in the past 30 days sits near 50% in July and August (50.1% and 48.3%) and 43.5% in September.

## 7. The in-scope table (hand-off to EDA_SDA_2)

`data/clean/lean/analysis_table.parquet` (it moves to `data/clean/` when this notebook replaces `EDA_SDA_1`), with the full descriptions in `scope_descriptions.parquet` (joined by `id`); the dictionary lists every column.

In [ ]:
# 7a-1 Get the in-scope table ready to save
# The one-hot contract_* and category_* columns (2a-3) were only for counting labels in 4e. The table keeps the labels
# in its contract_types and categories list columns, and EDA_SDA_2 loads every column it isn't told to skip, so
# leaving ~200 extra True/False columns in would hand it columns it never asked for.
one_hot_columns = [c for c in final.columns if c.startswith(("contract_", "category_"))]
final = final.drop(columns=one_hot_columns)

# save_final() builds its Parquet layout from an empty copy of the table, and pyarrow can't tell what type an empty
# text (object) column holds, so it types it "null" and then rejects the real values. Category columns it handles, as
# EDA_SDA_1 had them: so role_bucket and tripleten_program (text since 4c-2) go back to category.
final[["role_bucket", "tripleten_program"]] = final[["role_bucket", "tripleten_program"]].astype("category")
print(f"left out {len(one_hot_columns)} one-hot columns; {final.shape[1]} columns to save")

In [ ]:
# 7a-2 Save the in-scope table for EDA_SDA_2: the columns above, plus the text and list columns from the stream's rows (id,
# title, title_norm, job_title, desc_hash, contract_types with unified labels, tags, categories and the raw fields kept
# for EDA_SDA_2's feature screening), added part by part. EDA_SDA_2 reads only the columns it needs.
import pyarrow.parquet as pq
from src.rows_table import save_final

FINAL_PATH = LEAN_DIR / "analysis_table.parquet"
start = time.time()
save_final(final, FINAL_PATH)
print(f"saved {FINAL_PATH.name} in {time.time() - start:.0f}s ({FINAL_PATH.stat().st_size / 1e9:.2f} GB, "
      f"{len(pq.read_schema(FINAL_PATH).names)} columns); descriptions in {DESC_PATH.name}")

In [ ]:
# 7b Feature dictionary: every column of the in-scope table, its role and what it means (src/feature_dictionary.py)
from src.feature_dictionary import feature_dictionary

dictionary, undocumented = feature_dictionary(final, FINAL_PATH)
print("In-scope table: feature dictionary, by role (coverage = share of in-scope listings with a value)")
display(dictionary)
if undocumented:
    print("columns without a description:", undocumented)

In [ ]:
# 7c Check against EDA_SDA_1's hand-off table: the same rows and the same columns? (row counts and column names only)
# Expected differences: this table has 29K fewer rows (the three occupations dropped in 4c-1), and not the packed contract_bits and category_bits
ORIGINAL_PATH = CLEAN_DIR / "analysis_table.parquet"
if ORIGINAL_PATH.exists():
    original, lean = pq.ParquetFile(ORIGINAL_PATH), pq.ParquetFile(FINAL_PATH)
    print(f"rows: EDA_SDA_1 {original.metadata.num_rows:,}, this notebook {lean.metadata.num_rows:,}")
    print("columns only in EDA_SDA_1:", sorted(set(original.schema_arrow.names) - set(lean.schema_arrow.names)))
    print("columns only in this notebook:", sorted(set(lean.schema_arrow.names) - set(original.schema_arrow.names)))
else:
    print("EDA_SDA_1's table isn't on disk, so there is nothing to compare")

## 8. Findings

| | |
|:--|:--|
| **Data** | 13,415,388 listings in the file; 13,017,464 English (or unlabeled) and US (or no country); first seen March 30 to September 30, 2026 (the data's end date) |
| **After cleaning** | 13.0M listings, none removed; 1.4M reposts flagged (10.5%); 11.7M originals from 324K companies; 22.3% still open on September 30 |
| **In-scope table** | 879K listings (822K originals) from 69K companies in 11 role buckets (3 occupations that aren't a tech path dropped, 29K listings); 25.5% still open on September 30; 58% under a TripleTen program, 42% adjacent tech roles |

**Q1, how tech compares:** computer and mathematical is the best-paid (1st of 23) and most remote (1st) field, 4th in volume, but 22nd on entry-level roles and openness, and last on asking 2 years or fewer. The scope's design follows from it: hide what a newcomer can't get, rank the rest, and point to the open doors (IT support, business and data analysis, Data Analytics roles).

**Q2, which factors drive prioritization** (inside the scope)

- **Company volume is the strongest timing signal:** listings close in a median 12 days at the highest-volume companies (500+ earlier listings) vs. 45 at the lowest-volume (1 to 4), with the bands in between stepping down steadily (45, 34, 28, 12); every step is a real difference (Gehan-Wilcoxon test, 5c).
- **The window is short,** and it's what the model predicts: 29% of listings are gone within a week, median 24 days (5b).
- **Source and recruiters matter:** 15 days on LinkedIn or Taleo vs. 72 on Ashby; 11 days with a named recruiter, though only 1.3% of listings name one (5d, 5f).
- **Pay and openness vary most between roles:** $31 to $74 an hour, and openness from 9% (IT project management) to 44% (IT support) (5a, 5g).
- **Weak, or flags only:** remote wording shows no gap, reposts are mostly re-listings, and PredictLeads crawl artifacts barely move timing (5b, 5e, 5h).

**Q3, time information:** posting age works for every listing (`first_seen` lags the posting date by a median 1.4 days); day of week only through `posted` (41%); no seasonality from six months. The still-open rule should be last-seen based: the app's status filter keeps 26% of listings last seen 30+ days before September 30 (6c).

### Limits
- **Six months of history,** and listings first seen before March 30 are missing, so early dates undercount what was open (the date picker starts July 1).
- **We see listings, not hires:** openness is what listings ask for.
- **The text flags are unvalidated keyword rules,** so they stay labels and opt-in filters.
- **Section 3 is descriptive;** EDA_SDA_4 tests the gaps and confounders (staffing agencies are flagged in EDA_SDA_2).

**Handoff:** the in-scope table, its descriptions, the occupation summary and company activity go to **EDA_SDA_2**.